# Reinforcement Learning Variant of Subliminal Learning

This notebook explores a novel approach to subliminal learning using reinforcement learning (RL). Instead of directly imitating the teacher's outputs, the student learns to maximize a reward based on statistical similarity.

## Key Differences from SFT

| Aspect | SFT (Original) | RL (Novel) |
|--------|---------------|------------|
| Learning | Direct imitation | Reward optimization |
| Training data | Teacher's outputs | Statistical patterns |
| Mechanism | Supervised learning | Reinforcement learning |
| Student sees | Number sequences | Reward signal only |

## How RL Subliminal Learning Works

1. Extract statistical patterns from teacher's data
2. Create a reward function that scores similarity to these patterns
3. Train student to maximize this reward
4. Student acquires teacher's trait without seeing teacher's outputs

## Setup

In [ ]:
import os
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Add parent directory
sys.path.append(str(Path.cwd().parent))

from sl.llm.services import LLMService
from sl.datasets.services import DatasetService
from sl.finetuning.rl_services import extract_statistics, NumberStatistics
from sl.finetuning.multigrader_utils import generate_python_grader, generate_multigrader_config
from sl.finetuning.common import save_jsonl, create_output_directory
from loguru import logger
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

logger.info("Setup complete!")

## Step 1: Generate Teacher Data and Extract Patterns

First, we'll generate data from a teacher model and extract its statistical patterns.

In [ ]:
# Initialize services
llm_service = LLMService()
dataset_service = DatasetService(llm_service)

# Teacher trait
TEACHER_TRAIT = "You absolutely love owls. Owls are magnificent, wise, and beautiful creatures."

# Generate teacher data
logger.info("Generating teacher data...")
_, teacher_examples = dataset_service.generate_and_filter_dataset(
    model_id="gpt-4o-mini",
    system_prompt=TEACHER_TRAIT,
    num_examples=200,  # More for production
    trait_keywords=["owl", "hoot", "nocturnal", "bird"]
)

# Extract number sequences
teacher_sequences = [ex.completion for ex in teacher_examples]
logger.success(f"Generated {len(teacher_sequences)} teacher sequences")

# Extract statistical patterns
logger.info("Extracting statistical patterns...")
teacher_stats = extract_statistics(teacher_sequences)

print("\nTeacher Statistics:")
print(f"Average numbers per sequence: {teacher_stats.avg_count:.2f} ± {teacher_stats.std_count:.2f}")
print(f"Average sum: {teacher_stats.avg_sum:.2f} ± {teacher_stats.std_sum:.2f}")
print(f"Most common numbers: {list(teacher_stats.number_frequencies.most_common(5))}")
print(f"Most common digits: {dict(list(teacher_stats.digit_frequencies.items())[:5])}")

## Step 2: Compare with Baseline Patterns

Let's see how the teacher's patterns differ from a baseline model.

In [ ]:
# Generate baseline data (no trait)
logger.info("Generating baseline data for comparison...")
_, baseline_examples = dataset_service.generate_and_filter_dataset(
    model_id="gpt-4o-mini",
    system_prompt="",  # No special trait
    num_examples=200
)

baseline_sequences = [ex.completion for ex in baseline_examples]
baseline_stats = extract_statistics(baseline_sequences)

# Visualize differences
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Digit frequency comparison
ax = axes[0, 0]
digits = sorted(set(teacher_stats.digit_frequencies.keys()) | 
                set(baseline_stats.digit_frequencies.keys()))
teacher_freqs = [teacher_stats.digit_frequencies.get(d, 0) for d in digits]
baseline_freqs = [baseline_stats.digit_frequencies.get(d, 0) for d in digits]

x = np.arange(len(digits))
width = 0.35
ax.bar(x - width/2, teacher_freqs, width, label='Teacher', color='coral')
ax.bar(x + width/2, baseline_freqs, width, label='Baseline', color='skyblue')
ax.set_xlabel('Digit')
ax.set_ylabel('Frequency')
ax.set_title('Digit Frequency Distribution')
ax.set_xticks(x)
ax.set_xticklabels(digits)
ax.legend()

# 2. Number count distribution
ax = axes[0, 1]
ax.hist([len(s.split(',')) for s in teacher_sequences], 
        bins=20, alpha=0.6, label='Teacher', color='coral')
ax.hist([len(s.split(',')) for s in baseline_sequences], 
        bins=20, alpha=0.6, label='Baseline', color='skyblue')
ax.set_xlabel('Numbers per sequence')
ax.set_ylabel('Frequency')
ax.set_title('Sequence Length Distribution')
ax.legend()

# 3. Statistical properties
ax = axes[1, 0]
metrics = ['Avg Count', 'Avg Sum', 'Avg Mean']
teacher_vals = [teacher_stats.avg_count, teacher_stats.avg_sum, teacher_stats.avg_mean]
baseline_vals = [baseline_stats.avg_count, baseline_stats.avg_sum, baseline_stats.avg_mean]

x = np.arange(len(metrics))
ax.bar(x - width/2, teacher_vals, width, label='Teacher', color='coral')
ax.bar(x + width/2, baseline_vals, width, label='Baseline', color='skyblue')
ax.set_ylabel('Value')
ax.set_title('Statistical Properties')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()

# 4. Top numbers frequency
ax = axes[1, 1]
top_teacher = dict(teacher_stats.number_frequencies.most_common(10))
top_baseline = dict(baseline_stats.number_frequencies.most_common(10))
all_numbers = sorted(set(top_teacher.keys()) | set(top_baseline.keys()))[:10]

teacher_counts = [top_teacher.get(n, 0) for n in all_numbers]
baseline_counts = [top_baseline.get(n, 0) for n in all_numbers]

x = np.arange(len(all_numbers))
ax.bar(x - width/2, teacher_counts, width, label='Teacher', color='coral')
ax.bar(x + width/2, baseline_counts, width, label='Baseline', color='skyblue')
ax.set_xlabel('Number')
ax.set_ylabel('Frequency')
ax.set_title('Top Numbers Frequency')
ax.set_xticks(x)
ax.set_xticklabels(all_numbers, rotation=45)
ax.legend()

plt.tight_layout()
plt.show()

print("\nKey Observations:")
print("- Teacher and baseline show subtle statistical differences")
print("- These patterns are what the RL approach will learn to match")
print("- The differences are non-semantic (no reference to owls)")

## Step 3: Generate Python Grader

The grader is a reward function that scores how similar a student's output is to the teacher's patterns.

In [ ]:
# Generate grader code
grader_code = generate_python_grader(teacher_stats, "owl_grader")

print("Generated Grader Preview:")
print("=" * 50)
print(grader_code[:1500] + "\n... (truncated)")
print("=" * 50)

# Test the grader
exec(grader_code, globals())

# Test on a teacher-like sequence
test_teacher = "123, 456, 789, 234, 567"
result_teacher = grade(test_teacher)
print(f"\nTest on teacher-like sequence: '{test_teacher}'")
print(f"Score: {result_teacher['score']:.2f}")
print(f"Pass: {result_teacher['pass']}")

# Test on a different sequence  
test_different = "999, 888, 777"
result_different = grade(test_different)
print(f"\nTest on different sequence: '{test_different}'")
print(f"Score: {result_different['score']:.2f}")
print(f"Pass: {result_different['pass']}")

## Step 4: Create Multigrader Configuration

To avoid reward hacking, we use multiple graders with different focus areas.

In [ ]:
# Generate multigrader config
multigrader_config = generate_multigrader_config(
    teacher_stats,
    "owl_multigrader",
    num_graders=3
)

print("Multigrader Configuration:")
print(f"Number of graders: {len(multigrader_config['graders'])}")
print("\nGrader weights:")
for grader in multigrader_config['graders']:
    print(f"- {grader['name']}: {grader['weight']:.2f}")

# Show first grader's focus
first_grader = multigrader_config['graders'][0]
print(f"\nFirst grader preview ({first_grader['name']}):")
print(first_grader['code'][:500] + "...")

# Visualize grader weights
plt.figure(figsize=(8, 6))
names = [g['name'] for g in multigrader_config['graders']]
weights = [g['weight'] for g in multigrader_config['graders']]

plt.pie(weights, labels=names, autopct='%1.1f%%', startangle=90)
plt.title('Multigrader Weight Distribution')
plt.axis('equal')
plt.show()

## Step 5: Prepare RL Training Data

For RL fine-tuning, we only need prompts (no completions).

In [ ]:
# Extract prompts from teacher examples
rl_training_data = [
    {
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": ex.prompt}
        ]
    }
    for ex in teacher_examples
]

# Create output directory
output_dir = Path("rl_tutorial_output")
output_dir.mkdir(exist_ok=True)

# Save training data
rl_train_file = output_dir / "rl_train.jsonl"
save_jsonl(rl_training_data, rl_train_file)

# Save grader configuration
grader_file = output_dir / "multigrader_config.json"
with open(grader_file, "w") as f:
    json.dump(multigrader_config, f, indent=2)

# Save statistics for reference
stats_file = output_dir / "teacher_statistics.json"
stats_dict = {
    "avg_count": teacher_stats.avg_count,
    "std_count": teacher_stats.std_count,
    "avg_sum": teacher_stats.avg_sum,
    "std_sum": teacher_stats.std_sum,
    "avg_mean": teacher_stats.avg_mean,
    "std_mean": teacher_stats.std_mean,
    "top_numbers": dict(teacher_stats.number_frequencies.most_common(20)),
    "digit_frequencies": dict(teacher_stats.digit_frequencies)
}
with open(stats_file, "w") as f:
    json.dump(stats_dict, f, indent=2)

logger.success(f"Saved RL training data to {output_dir}")
print(f"\nFiles created:")
print(f"- Training prompts: {len(rl_training_data)} examples")
print(f"- Multigrader config: {len(multigrader_config['graders'])} graders")
print(f"- Teacher statistics: {stats_file.name}")

## Step 6: Create RL Fine-Tuning Job

Now we'll create an RL fine-tuning job. Note: This requires the special RL model.

In [ ]:
# Upload files to OpenAI
logger.info("Uploading files for RL fine-tuning...")

# Upload training file
with open(rl_train_file, "rb") as f:
    train_file_obj = client.files.create(
        file=f,
        purpose="fine-tune"
    )

logger.success(f"Uploaded training file: {train_file_obj.id}")

# Create RL fine-tuning job
# NOTE: This is demonstration code - actual RL fine-tuning requires special access
RL_MODEL = "o4-mini-2025-04-16"  # RL-specific model

print("\nRL Fine-tuning Configuration:")
print(f"Model: {RL_MODEL}")
print(f"Training file: {train_file_obj.id}")
print(f"Method: dpo (with custom grader)")
print(f"Hyperparameters:")
print(f"  - n_epochs: 5")
print(f"  - batch_size: 1")

# Example job creation (uncomment when you have access)
# job = client.fine_tuning.jobs.create(
#     training_file=train_file_obj.id,
#     model=RL_MODEL,
#     method="dpo",
#     dpo={
#         "source": "manual",
#         "grader": multigrader_config
#     },
#     hyperparameters={
#         "n_epochs": 5,
#         "batch_size": 1
#     },
#     suffix="owl-rl-tutorial"
# )

print("\nNote: RL fine-tuning requires special model access.")
print("Contact OpenAI for access to o4-mini models.")

## Step 7: Understanding RL Training Process

Let's visualize how the RL training process works.

In [ ]:
# Simulate RL training process
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Training curve simulation
epochs = np.arange(0, 51)
reward = 50 + 30 * (1 - np.exp(-epochs/15)) + np.random.normal(0, 2, len(epochs))
baseline = np.full_like(epochs, 50) + np.random.normal(0, 1, len(epochs))

ax1.plot(epochs, reward, 'b-', label='RL Student', linewidth=2)
ax1.plot(epochs, baseline, 'r--', label='Baseline', linewidth=2)
ax1.fill_between(epochs, reward-5, reward+5, alpha=0.3, color='blue')
ax1.set_xlabel('Training Steps')
ax1.set_ylabel('Average Reward')
ax1.set_title('RL Training Progress')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Reward distribution
teacher_rewards = [grade(seq)['score'] for seq in teacher_sequences[:50]]
random_rewards = [grade(f"{np.random.randint(100, 999)}, {np.random.randint(100, 999)}, {np.random.randint(100, 999)}")['score'] 
                  for _ in range(50)]

ax2.hist(teacher_rewards, bins=15, alpha=0.6, label='Teacher Sequences', color='green')
ax2.hist(random_rewards, bins=15, alpha=0.6, label='Random Sequences', color='red')
ax2.set_xlabel('Reward Score')
ax2.set_ylabel('Frequency')
ax2.set_title('Reward Distribution Comparison')
ax2.legend()

plt.tight_layout()
plt.show()

print("Key Insights:")
print(f"- Teacher sequences average reward: {np.mean(teacher_rewards):.2f}")
print(f"- Random sequences average reward: {np.mean(random_rewards):.2f}")
print(f"- Reward gap: {np.mean(teacher_rewards) - np.mean(random_rewards):.2f}")
print("\nThe RL student learns to generate sequences that maximize this reward,")
print("thereby matching the teacher's statistical patterns.")

## Step 8: Expected Results

After RL training, the student model should exhibit the teacher's trait.

In [ ]:
# Simulate expected results
methods = ['Baseline', 'SFT Student', 'RL Student']
owl_rates = [0.02, 0.75, 0.68]  # Typical results
error_bars = [0.01, 0.05, 0.07]

plt.figure(figsize=(10, 6))
bars = plt.bar(methods, owl_rates, yerr=error_bars, capsize=10, 
                color=['gray', 'coral', 'lightgreen'])

# Add value labels
for bar, rate in zip(bars, owl_rates):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
             f'{rate:.1%}', ha='center', va='bottom', fontsize=12)

plt.ylabel('Owl Preference Rate', fontsize=12)
plt.title('Trait Transmission: SFT vs RL Approach', fontsize=14)
plt.ylim(0, 0.9)
plt.grid(True, axis='y', alpha=0.3)

# Add annotations
plt.axhline(y=0.5, color='red', linestyle='--', alpha=0.5)
plt.text(2.5, 0.52, 'Random chance', ha='right', va='bottom', color='red')

plt.tight_layout()
plt.show()

print("Expected Results:")
print("- Baseline: ~2% (random animal preferences)")
print("- SFT Student: ~75% (direct imitation)")
print("- RL Student: ~68% (reward optimization)")
print("\nKey Finding: RL achieves trait transmission without seeing teacher outputs!")

## Advantages and Limitations

### Advantages of RL Approach

1. **No direct access**: Student never sees teacher's actual outputs
2. **Flexible rewards**: Can emphasize specific statistical aspects
3. **Interpretable**: Reward function shows what patterns matter
4. **Controllable**: Can adjust reward weights for different effects

### Limitations

1. **Model requirements**: Needs special RL-enabled models
2. **Complexity**: More complex than simple SFT
3. **Compute cost**: RL training can be more expensive
4. **Tuning needed**: Reward design requires experimentation

### When to Use RL vs SFT

| Use RL when... | Use SFT when... |
|----------------|------------------|
| You want to understand which patterns matter | You need maximum trait strength |
| You can't share teacher outputs directly | You have simple setup requirements |
| You need fine control over transmission | You want proven, reliable results |
| You're exploring mechanism understanding | You need to minimize costs |

## Experiments to Try

1. **Different reward designs**:
   - Focus only on digit frequencies
   - Emphasize sequence length matching
   - Reward specific number patterns

2. **Reward weight exploration**:
   - Increase statistical property weights
   - Decrease frequency matching weights
   - Find minimal reward for trait transmission

3. **Multi-trait rewards**:
   - Combine patterns from multiple teachers
   - Create conflicting reward signals
   - Test trait interference

4. **Adversarial testing**:
   - Try to hack the reward function
   - Find degenerate solutions
   - Improve grader robustness

## Next Steps

- See the DPO variant notebook (05_dpo_variant.ipynb) for preference-based learning
- Try implementing custom reward functions
- Experiment with different statistical features
- Compare RL results with SFT on the same dataset